# EU Industrial Process Heat by Temperature Band (2024)

Multi-sector heat demand analysis built on the bottom-up pipeline in
`heat_demand/facilities/` (every assumption documented in `METHODS.md` there):

1. `export_facilities_2024.py` — Climate TRACE facility master → EU manufacturing facilities
2. `combine_hotmaps_2014.py` — spatial match against Fraunhofer hotmaps Industrial Database
3. `merge_production.py` — production source selection + heat estimates (SEC / emissions inversion)
4. `heat_temperature_bands.py` — temperature-band split via hotmaps industry benchmarks
5. `apply_dedup.py` — executes the reviewed duplicate-site decisions (80 rows removed)

**Data source:** `facilities_2024_eu_deduplicated.csv` (pipeline step 5 output).
**Heat basis:** `useful_heat_tj` (METHODS §3–4 estimates). The benchmark-derived band columns
are rescaled to that basis, so band totals sum exactly to `useful_heat_tj` per facility.
**Scope notes:** iron & steel is final-consumption scope (coke ovens and blast furnaces
excluded — Eurostat books them in the transformation sector); food heat is rescaled per
country to the useful-energy reference — Climate TRACE food emissions serve only as
within-country allocation weights (METHODS §3).

Run the five scripts above first to refresh the data.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Support running from project root or heat_demand/analysis_outputs/
_cwd = Path.cwd()
if (_cwd / "heat_demand" / "facilities").exists():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "facilities").exists():   # analysis_outputs/
    PROJECT_ROOT = _cwd.parent.parent
else:
    PROJECT_ROOT = _cwd.parent

FACILITIES_DIR = PROJECT_ROOT / "heat_demand" / "facilities"
output_dir = PROJECT_ROOT / "heat_demand" / "analysis_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

BANDS = ["below100C", "100C-200C", "200C-500C", "500C-1000C", "above1000C"]
BAND_LABELS = {
    "below100C": "<100°C", "100C-200C": "100–200°C", "200C-500C": "200–500°C",
    "500C-1000C": "500–1000°C", "above1000C": ">1000°C",
}
# Ordinal single-hue ramp, light→dark = low→high temperature (validated: CVD-safe)
BAND_COLORS = {
    "<100°C": "#86b6ef", "100–200°C": "#5598e7", "200–500°C": "#2a78d6",
    "500–1000°C": "#1c5cab", ">1000°C": "#104281",
}
SURFACE = "#fcfcfb"
BAND_COLS = [f"heat_{b}_tj" for b in BANDS]

In [2]:
# Load banded, deduplicated facility data (pipeline step 5 output)
bands = pd.read_csv(FACILITIES_DIR / "facilities_2024_eu_deduplicated.csv")
matched_hm = pd.read_csv(FACILITIES_DIR / "facilities_matched_hotmaps_2014.csv")

# Heat basis = useful_heat_tj; rescale the benchmark band columns onto it so
# the five bands sum exactly to useful_heat_tj for every facility.
banded = bands[bands["bench_heat_tj"].gt(0) & bands["useful_heat_tj"].notna()].copy()
_shares = banded[BAND_COLS].div(banded[BAND_COLS].sum(axis=1), axis=0)
for _c in BAND_COLS:
    banded[_c] = _shares[_c] * banded["useful_heat_tj"]

print(f"Facilities with banded heat: {len(banded)} of {len(bands)}")
print(f"Total useful heat: {banded['useful_heat_tj'].sum() / 3600:,.0f} TWh_th/yr")
banded.groupby("heat_method")["useful_heat_tj"].agg(sites="count", heat_tj="sum")

Facilities with banded heat: 2190 of 2413
Total useful heat: 632 TWh_th/yr


,sites,heat_tj
heat_method,,
emissions_fuel_inversion,1479,4.606653e+05
production_sec,711,1.814182e+06


## Heat demand by temperature band and industry

In [3]:
by_sub = banded.groupby("subsector")[BAND_COLS].sum() / 3600  # TWh
by_sub.columns = [BAND_LABELS[b] for b in BANDS]
by_sub["total"] = by_sub.sum(axis=1)
by_sub = by_sub.sort_values("total", ascending=True)  # ascending: largest on top of h-bar

band_long = (
    by_sub.drop(columns="total")
    .reset_index()
    .melt(id_vars="subsector", var_name="band", value_name="twh")
)

fig_bands = px.bar(
    band_long,
    y="subsector", x="twh", color="band", orientation="h",
    color_discrete_map=BAND_COLORS,
    category_orders={
        "band": list(BAND_COLORS.keys()),
        "subsector": by_sub.index.tolist(),
    },
    labels={"twh": "Useful heat (TWh_th/yr)", "subsector": "", "band": "Temperature band"},
    title="EU industrial process heat by temperature band (bottom-up, 2024)",
)
# 2px surface gap between stacked segments; recessive grid
fig_bands.update_traces(marker_line_color=SURFACE, marker_line_width=1.5)
fig_bands.update_layout(
    height=480, width=1000, plot_bgcolor=SURFACE, paper_bgcolor=SURFACE,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    xaxis=dict(gridcolor="#e8e8e6"), yaxis=dict(gridcolor=SURFACE),
)
# selective direct labels: totals at bar ends only
for sub, total in by_sub["total"].items():
    fig_bands.add_annotation(
        y=sub, x=total, text=f"{total:,.0f}", showarrow=False,
        xanchor="left", xshift=6, font=dict(size=11, color="#52514e"),
    )
fig_bands.show()

In [4]:
# Band shares within each industry (composition, not magnitude)
share_long = band_long.copy()
share_long["share"] = share_long["twh"] / share_long.groupby("subsector")["twh"].transform("sum")

fig_shares = px.bar(
    share_long,
    y="subsector", x="share", color="band", orientation="h",
    color_discrete_map=BAND_COLORS,
    category_orders={
        "band": list(BAND_COLORS.keys()),
        "subsector": by_sub.index.tolist(),
    },
    labels={"share": "Share of heat demand", "subsector": "", "band": "Temperature band"},
    title="Temperature-band composition by industry",
)
fig_shares.update_traces(marker_line_color=SURFACE, marker_line_width=1.5)
fig_shares.update_layout(
    height=480, width=1000, plot_bgcolor=SURFACE, paper_bgcolor=SURFACE,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    xaxis=dict(tickformat=".0%", gridcolor="#e8e8e6"),
)
fig_shares.show()

# Table view (relief for low-contrast ramp steps)
band_table = by_sub.iloc[::-1].copy()
band_table.loc["EU total"] = band_table.sum()
band_table.round(1)

,<100°C,100–200°C,200–500°C,500–1000°C,>1000°C,total
subsector,,,,,,
iron-and-steel,0.0,0.1,13.0,67.0,65.3,145.4
food-beverage-tobacco,53.6,38.7,8.2,10.0,0.0,110.5
cement,0.0,0.0,10.8,65.0,32.5,108.4
petrochemical-steam-cracking,0.0,0.0,0.0,82.6,0.0,82.6
pulp-and-paper,1.1,51.4,1.1,0.4,0.0,54.1
chemicals,5.7,7.5,0.0,16.7,16.8,46.7
glass,0.7,7.3,10.6,7.7,9.7,36.1
lime,0.0,0.0,0.0,8.1,12.2,20.3
textiles-leather-apparel,9.6,6.1,1.7,0.0,0.0,17.5


In [5]:
# Heat by temperature band for the 12 largest countries
by_ctry = banded.groupby("iso3_country")[BAND_COLS].sum() / 3600
by_ctry["total"] = by_ctry.sum(axis=1)
top12 = by_ctry.sort_values("total", ascending=True).tail(12)

ctry_long = (
    top12.drop(columns="total")
    .rename(columns={c: BAND_LABELS[b] for c, b in zip(BAND_COLS, BANDS)})
    .reset_index()
    .melt(id_vars="iso3_country", var_name="band", value_name="twh")
)
fig_ctry = px.bar(
    ctry_long,
    y="iso3_country", x="twh", color="band", orientation="h",
    color_discrete_map=BAND_COLORS,
    category_orders={
        "band": list(BAND_COLORS.keys()),
        "iso3_country": top12.index.tolist(),
    },
    labels={"twh": "Useful heat (TWh_th/yr)", "iso3_country": "", "band": "Temperature band"},
    title="Process heat by temperature band — top 12 countries",
)
fig_ctry.update_traces(marker_line_color=SURFACE, marker_line_width=1.5)
fig_ctry.update_layout(
    height=480, width=1000, plot_bgcolor=SURFACE, paper_bgcolor=SURFACE,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    xaxis=dict(gridcolor="#e8e8e6"),
)
fig_ctry.show()

## Cross-checks — bottom-up vs independent estimates

Three checks of increasing altitude:

| # | Comparison | What it tests |
|---|---|---|
| 1 | Facility fuel energy vs **hotmaps reported fuel demand** (matched sites) | site-level plausibility |
| 2 | Sector totals vs **two top-downs**: Eurostat final energy × heat share, and the EU27 **useful energy demand** reference | aggregate coverage |
| 3 | METHODS §4 SECs vs **hotmaps benchmark** SECs (internal) | method sensitivity |

### Cross-check 1 — facility level vs hotmaps fuel demand

For high/medium-confidence matches, compare our estimated fuel energy for heat against the
fuel demand hotmaps reports per site (GWh → TJ). Ratio ≈ 1 means the two independent
bottom-up estimates agree; systematic offsets reveal boundary differences (e.g. our BF/BOF
recipe covers the whole steel route; many hotmaps steel entries are single process sites).

In [6]:
SUBSECTOR_COLORS = {  # fixed categorical order (validated palette slots)
    "cement": "#2a78d6", "iron-and-steel": "#1baf7a",
    "pulp-and-paper": "#eda100", "glass": "#008300",
}
cc1 = banded.merge(matched_hm[["source_id", "Fuel_Demand"]], on="source_id", how="left")
cc1 = cc1[
    cc1["match_confidence"].isin(["high", "medium"])
    & cc1["Fuel_Demand"].gt(0)
    & cc1["fuel_energy_tj"].gt(0)
].copy()
cc1["hotmaps_fuel_tj"] = cc1["Fuel_Demand"] * 3.6  # GWh -> TJ
cc1["ratio_ours_over_hotmaps"] = cc1["fuel_energy_tj"] / cc1["hotmaps_fuel_tj"]

lim = [
    min(cc1["hotmaps_fuel_tj"].min(), cc1["fuel_energy_tj"].min()) * 0.7,
    max(cc1["hotmaps_fuel_tj"].max(), cc1["fuel_energy_tj"].max()) * 1.4,
]
fig_cc1 = px.scatter(
    cc1,
    x="hotmaps_fuel_tj", y="fuel_energy_tj", color="subsector",
    color_discrete_map=SUBSECTOR_COLORS,
    category_orders={"subsector": list(SUBSECTOR_COLORS.keys())},
    hover_name="source_name",
    hover_data={"hotmaps_fuel_tj": ":,.0f", "fuel_energy_tj": ":,.0f",
                "ratio_ours_over_hotmaps": ":.2f", "match_confidence": True},
    log_x=True, log_y=True,
    labels={"hotmaps_fuel_tj": "hotmaps reported fuel demand (TJ/yr)",
            "fuel_energy_tj": "This study — fuel energy for heat (TJ/yr)"},
    title="Facility-level cross-check: this study vs hotmaps fuel demand",
)
fig_cc1.add_trace(go.Scatter(
    x=lim, y=lim, mode="lines", name="1:1",
    line=dict(color="#52514e", width=1, dash="dash"),
))
fig_cc1.update_traces(marker=dict(size=9, opacity=0.75,
                                  line=dict(color=SURFACE, width=1)),
                      selector=dict(mode="markers"))
fig_cc1.update_layout(
    height=560, width=880, plot_bgcolor=SURFACE, paper_bgcolor=SURFACE,
    xaxis=dict(gridcolor="#e8e8e6"), yaxis=dict(gridcolor="#e8e8e6"),
)
fig_cc1.show()

cc1_stats = cc1.groupby("subsector")["ratio_ours_over_hotmaps"].agg(
    n="count", median="median",
    p10=lambda s: s.quantile(0.1), p90=lambda s: s.quantile(0.9),
).round(2)
cc1_stats

,n,median,p10,p90
subsector,,,,
cement,156,0.57,0.28,1.00
glass,148,1.12,0.60,1.12
iron-and-steel,74,3.59,0.21,16.41
pulp-and-paper,19,1.91,0.81,46.38


### Cross-check 2 — sector totals vs two independent top-downs

Two references bracket the bottom-up:

1. **Eurostat × heat share** — EU27 *final energy* consumption by industry sector
   (`nrg_bal_s`, ~2022, PJ) × an assumed process-heat share (Fraunhofer ISI et al. 2016,
   Heat Roadmap Europe). Final energy is fuel input, so this is the **upper** yardstick.
   *Order-of-magnitude constants — replace with an exact Eurostat pull for publication.*
2. **Useful energy demand (EU27)** — `References/industry_useful_energy_demand_EU27.csv`;
   heat = the four *non-electric process heat* bands + *steam (non-electric boilers)*,
   electricity rows excluded to match our fuel-based estimates. Useful energy nets out
   conversion losses, so it is the **lower** yardstick.

**Category matching** (the reference splits sectors differently than Eurostat NACE):
*Non-metallic minerals* (NACE 23) covers cement, lime, glass, ceramics and bricks — matched
to the reference's *Cement* + *Ceramics and Glass* columns (lime has no column of its own);
*Iron & steel* = *Primary* + *Secondary Steel*; the rest map one-to-one. Reference sectors
with no facility coverage (Transport Equipment, Machinery, Wood) are not compared.

Interpretation of coverage:
- **< 100% vs Eurostat** expected everywhere — Climate TRACE covers large point sources
  only, and sector boundaries are wider than ours (ceramics/bricks not modelled). Iron &
  steel is *also* < 100% by construction now: the BF/BOF recipe keeps only sinter + rolling
  (~4.8 GJ/t), because coke ovens and blast furnaces sit in Eurostat's transformation
  sector, not industry final energy.
- **Between the two yardsticks is the healthy band** — our kiln sectors use eff = 1.0
  (fuel = heat by convention), which reads high against a useful-energy basis: cement/lime
  and steel land above the useful-energy reference but below Eurostat.
- **Food ≈ 107% vs useful energy is by construction** — food heat is rescaled per country
  to that reference (METHODS §3; the residual is GBR, scaled with the EU-26 aggregate
  factor but absent from the EU27 reference). Against Eurostat food now reads ~58%; the
  two yardsticks disagree by ~1.9× for this sector.
- **Paper low against both** (~21% / ~27%) reflects biomass-dominated heat (black liquor
  supplies ~60% of mill heat) and partial mill coverage (~80 of ~900 mills).

In [7]:
TOP_DOWN_REFERENCES = {
    # sector: EU27 final energy (PJ, Eurostat nrg_bal_s ~2022, approximate),
    #         process-heat share of final energy, mapped bottom-up subsectors
    "Iron & steel": dict(final_energy_pj=1150, heat_share=0.85,
                         subsectors=["iron-and-steel"]),
    "Non-metallic minerals": dict(final_energy_pj=1300, heat_share=0.85,
                                  subsectors=["cement", "lime", "glass"]),
    "Chemical & petrochemical": dict(final_energy_pj=1850, heat_share=0.70,
                                     subsectors=["chemicals", "petrochemical-steam-cracking"]),
    "Paper, pulp & printing": dict(final_energy_pj=1250, heat_share=0.75,
                                   subsectors=["pulp-and-paper"]),
    "Food, beverages & tobacco": dict(final_energy_pj=1150, heat_share=0.60,
                                      subsectors=["food-beverage-tobacco"]),
    "Non-ferrous metals": dict(final_energy_pj=240, heat_share=0.60,
                               subsectors=["aluminum"]),
    "Textile & leather": dict(final_energy_pj=130, heat_share=0.50,
                              subsectors=["textiles-leather-apparel"]),
}

# Second, independent top-down: EU27 useful energy demand
# (References/industry_useful_energy_demand). Heat = four non-electric
# process-heat bands + steam boilers; electricity rows excluded to match
# the fuel-based bottom-up. Non-metallic minerals = Cement + Ceramics and
# Glass (NACE 23 spans cement, lime, glass, ceramics, bricks); steel =
# Primary + Secondary.
UE_HEAT_ROWS = [
    "Non-electric process heat (<100 C)", "Non-electric process heat (100-400 C)",
    "Non-electric process heat (400-1000 C)", "Non-electric process heat (>1000 C)",
    "Steam (non-electric boilers)",
]
UE_SECTOR_COLS = {
    "Iron & steel": ["Primary Steel (TWh)", "Secondary Steel (TWh)"],
    "Non-metallic minerals": ["Cement (TWh)", "Ceramics and Glass (TWh)"],
    "Chemical & petrochemical": ["Chemicals (TWh)"],
    "Paper, pulp & printing": ["Pulp and Paper (TWh)"],
    "Food, beverages & tobacco": ["Food, Beverages and Tobacco (TWh)"],
    "Non-ferrous metals": ["Non-ferrous Metals (TWh)"],
    "Textile & leather": ["Textiles and Leather (TWh)"],
}
ue = pd.read_csv(PROJECT_ROOT / "References" / "industry_useful_energy_demand"
                 / "industry_useful_energy_demand_EU27.csv")
ue_heat = ue[ue["energy_demand_type"].isin(UE_HEAT_ROWS)]
ue_twh = {sector: float(ue_heat[cols].sum().sum())
          for sector, cols in UE_SECTOR_COLS.items()}

sub_totals_twh = banded.groupby("subsector")["useful_heat_tj"].sum() / 3600
cc2 = pd.DataFrame([
    {
        "sector": sector,
        "bottom_up_twh": sub_totals_twh.reindex(ref["subsectors"]).sum(),
        "top_down_heat_twh": ref["final_energy_pj"] / 3.6 * ref["heat_share"],
        "useful_energy_twh": ue_twh[sector],
    }
    for sector, ref in TOP_DOWN_REFERENCES.items()
])
cc2["coverage_vs_eurostat_pct"] = 100 * cc2["bottom_up_twh"] / cc2["top_down_heat_twh"]
cc2["coverage_vs_useful_pct"] = 100 * cc2["bottom_up_twh"] / cc2["useful_energy_twh"]
cc2 = cc2.sort_values("top_down_heat_twh", ascending=True)

cc2_long = cc2.melt(
    id_vars=["sector"],
    value_vars=["bottom_up_twh", "useful_energy_twh", "top_down_heat_twh"],
    var_name="estimate", value_name="twh",
)
EST_LABELS = {
    "bottom_up_twh": "Bottom-up (this study)",
    "top_down_heat_twh": "Top-down (Eurostat × heat share)",
    "useful_energy_twh": "Top-down (useful energy, EU27 ref)",
}
cc2_long["estimate"] = cc2_long["estimate"].map(EST_LABELS)
fig_cc2 = px.bar(
    cc2_long,
    y="sector", x="twh", color="estimate", orientation="h", barmode="group",
    color_discrete_map={
        "Bottom-up (this study)": "#2a78d6",
        "Top-down (useful energy, EU27 ref)": "#1baf7a",
        "Top-down (Eurostat × heat share)": "#eda100",
    },
    category_orders={"estimate": list(EST_LABELS.values())},
    labels={"twh": "Process heat (TWh_th/yr)", "sector": "", "estimate": ""},
    title="Bottom-up vs two top-down references, by sector",
)
fig_cc2.update_traces(marker_line_color=SURFACE, marker_line_width=1.5)
fig_cc2.update_layout(
    height=520, width=980, plot_bgcolor=SURFACE, paper_bgcolor=SURFACE,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    xaxis=dict(gridcolor="#e8e8e6"),
)
fig_cc2.show()

cc2.sort_values("top_down_heat_twh", ascending=False).round(1)

,sector,bottom_up_twh,top_down_heat_twh,useful_energy_twh,coverage_vs_eurostat_pct,coverage_vs_useful_pct
2,Chemical & petrochemical,129.2,359.7,172.5,35.9,74.9
1,Non-metallic minerals,164.8,306.9,182.0,53.7,90.6
0,Iron & steel,145.4,271.5,99.5,53.5,146.1
3,"Paper, pulp & printing",54.1,260.4,203.4,20.8,26.6
4,"Food, beverages & tobacco",110.5,191.7,103.0,57.7,107.3
5,Non-ferrous metals,10.5,40.0,22.2,26.2,47.2
6,Textile & leather,17.5,18.1,11.5,96.7,151.5


### Cross-check 3 — internal method sensitivity (two SEC sources)

For production-covered facilities we have two independent heat estimates:
`useful_heat_tj` (METHODS.md §4 SECs — IEA/GNR/EuLA/CEPI/worldsteel) and `bench_heat_tj`
(hotmaps benchmark process recipes). Their ratio shows how sensitive results are to the SEC
source. Petrochemical stands out by construction: the benchmark ethylene SEC (35.9 GJ/t)
includes byproduct fuel-gas firing; METHODS §4 (17 GJ/t) counts external fuel only.

In [8]:
cc3_src = bands[
    (bands["heat_method"] == "production_sec")
    & bands["useful_heat_tj"].gt(0) & bands["bench_heat_tj"].gt(0)
]
cc3 = pd.DataFrame({
    "n": cc3_src.groupby("subsector").size(),
    "methods_sec_twh": cc3_src.groupby("subsector")["useful_heat_tj"].sum() / 3600,
    "benchmark_sec_twh": cc3_src.groupby("subsector")["bench_heat_tj"].sum() / 3600,
})
cc3["ratio_benchmark_over_methods"] = cc3["benchmark_sec_twh"] / cc3["methods_sec_twh"]
cc3.sort_values("benchmark_sec_twh", ascending=False).round(2)

,n,methods_sec_twh,benchmark_sec_twh,ratio_benchmark_over_methods
subsector,,,,
petrochemical-steam-cracking,36,82.56,174.35,2.11
iron-and-steel,112,145.36,157.89,1.09
cement,198,108.36,104.30,0.96
chemicals,48,46.68,61.71,1.32
glass,148,36.14,43.64,1.21
pulp-and-paper,79,54.05,42.88,0.79
lime,62,20.31,16.70,0.82
aluminum,28,10.48,11.25,1.07


### Takeaways

- **~30% of covered EU industrial heat (≈189 of 632 TWh_th/yr) is below 200°C** (food,
  paper, textiles dominated) — addressable by current steam-output heat batteries; the
  ~63% above 500°C sits in steel, cement, lime, glass and crackers.
- **The bottom-up sits between the two top-down yardsticks in most sectors** — e.g.
  iron & steel at 54% of Eurostat × heat share but 146% of the useful-energy reference,
  non-metallic minerals at 54% / 91% — consistent with our fuel-based, large-point-source
  scope. Food is pinned to the useful-energy reference by construction (per-country
  rescaling after the Climate TRACE mis-allocation — Portugal was 14×; METHODS §3),
  and reads ~58% against Eurostat. Paper is low
  against both (21% / 27%): black liquor supplies ~60% of mill heat and coverage is
  ~80 of ~900 mills.
- Facility-level agreement with hotmaps is good for **glass (median ratio 1.12)** and
  reasonable for **cement (0.57, boundary: hotmaps includes non-kiln fuel)**; steel
  scatter is wide (median 3.6, p10–p90 of 0.2–16) — the final-consumption SEC (sinter +
  rolling only) no longer matches hotmaps' whole-site fuel, so treat individual steel
  sites with care.
- SEC-source sensitivity is **±30% for most sectors** (2.1× for petrochemical by
  definition — the benchmark ethylene SEC includes byproduct fuel-gas firing).

In [9]:
# Export summary tables
by_sub.iloc[::-1].round(2).to_csv(output_dir / "heat_by_band_subsector.csv")
by_ctry.sort_values("total", ascending=False).round(2).to_csv(output_dir / "heat_by_band_country.csv")
cc1_stats.to_csv(output_dir / "crosscheck1_hotmaps_fuel_ratios.csv")
cc2.round(2).to_csv(output_dir / "crosscheck2_topdown_coverage.csv", index=False)
cc3.round(3).to_csv(output_dir / "crosscheck3_sec_sensitivity.csv")
print(f"Summary tables written to {output_dir}")

Summary tables written to /Users/annayuen/Desktop/IECC/industrial_decarb/SEC_PPI_IHB_potential/heat_demand/analysis_outputs
